# Egress Index, step 3 of 3: elevationThe flood hazard needs to know how low each road sits. This samples a DEMonto the graph and writes two numbers back into it.**In:** `graph_santa_rosa.json` from step 2, plus 3DEP elevation via`py3dep`.**Out:** the same file, with `elev` on every node and `emin` on every edge.`emin` is the minimum elevation sampled along the pavement itself, at about9 m spacing, rather than the elevation at either end. That is the underpasscatch: a road that dips under a rail line floods at the dip, not at thecorners, and endpoint elevations miss it entirely.**Runtime:** a few minutes.Part of https://github.com/jerrod-lessel/egress-index

## 1. DEM diagnostic: prove the elevation data is sane

Before sampling thousands of nodes and edges, we pull **one** small elevation
tile and check it looks right. Cheap first, expensive later.

**What a DEM is:** a Digital Elevation Model, a grid where every pixel holds a
ground height in meters. We get ours from **USGS 3DEP** (the national elevation
dataset) through the `py3dep` library, at 10 m resolution.

**The one gotcha:** `py3dep.get_dem` always returns the tile in **EPSG:5070**,
an Albers projection measured in **meters**, not lon/lat degrees. So the grid's
x and y are meters. That is good news for later (edge lengths for sampling are
already metric), but it means we must convert each node's lon/lat into 5070
before we can look up its pixel.

**Two ways we check the same elevation:**
1. Sample the DEM tile at the node's location.
2. Ask 3DEP directly for that exact point (`elevation_bycoords`).
If those two land close together, our raster sampling is trustworthy.

**What healthy output looks like:** Santa Rosa sits roughly 15 to 300 m above
sea level. Elevation min/max in that ballpark, pixel size near 10 m, and the two
elevation checks agreeing within a meter or two means we are good to proceed. A
wild negative min (like -9999) would mean a nodata fill we will mask in Cell 2.

In [3]:
# ── Cell 1: DEM diagnostic ──────────────────────────────────────────────
# Pull ONE small elevation tile for Santa Rosa and prove it is sane before
# we sample all 6,749 nodes and 16,254 edges. No big run happens here.

!pip install -q py3dep

import json, requests
import numpy as np
import rioxarray                      # registers the .rio accessor used below
import py3dep
from pyproj import Transformer

# ── 1. Load the routing graph so we know where the nodes are ────────────
# If the repo is public this raw URL just works. If the fetch fails (private
# repo), the except block lets you upload graph_santa_rosa.json by hand.
GRAPH_URL = "https://raw.githubusercontent.com/jerrod-lessel/egress-index/main/graph_santa_rosa.json"
try:
    graph = requests.get(GRAPH_URL, timeout=30).json()
    print("Graph loaded from GitHub raw.")
except Exception as e:
    print("Raw fetch failed, upload the file instead:", e)
    from google.colab import files
    up = files.upload()                         # pick graph_santa_rosa.json
    graph = json.loads(next(iter(up.values())))

nodes = graph["nodes"]                           # array index == node id
lons  = np.array([n["x"] for n in nodes])        # longitude, 5 decimal places
lats  = np.array([n["y"] for n in nodes])        # latitude
print(f"{len(nodes):,} nodes loaded.")

# ── 2. Bounding box around every node, with a little breathing room ──────
# 0.01 degrees is roughly 1 km. The pad makes sure roads that bend slightly
# past the outermost node still sit on top of real elevation data.
PAD = 0.01
bbox = (lons.min() - PAD, lats.min() - PAD,      # west, south
        lons.max() + PAD, lats.max() + PAD)      # east, north
print("bbox (W,S,E,N):", tuple(round(v, 4) for v in bbox))

# ── 3. Pull the DEM at 10 m ─────────────────────────────────────────────
# get_dem always returns the grid in EPSG:5070 (Albers, METERS), regardless
# of the bbox CRS. So the tile's x/y are meters, and we convert lon/lat to
# 5070 before sampling (step 4).
dem = py3dep.get_dem(bbox, resolution=10, crs=4326)

print("\n── DEM summary ──")
print("CRS      :", dem.rio.crs)                                   # expect EPSG:5070
print("shape    :", dem.shape, "(rows, cols)")
print("pixel    :", round(abs(float(dem.x[1] - dem.x[0])), 2), "m")# expect ~10
print("elev min :", round(float(dem.min()), 1), "m")
print("elev max :", round(float(dem.max()), 1), "m")
print("elev mean:", round(float(dem.mean()), 1), "m")

# ── 4. Spot check two nodes against the raster ──────────────────────────
# Reproject their lon/lat into the DEM's 5070 meters, grab the nearest pixel.
to5070 = Transformer.from_crs(4326, 5070, always_xy=True)          # (lon,lat)->(x,y)
check_ids = [0, len(nodes) // 2]                                   # first + middle node

print("\n── Node spot check (raster) ──")
for i in check_ids:
    lon, lat = nodes[i]["x"], nodes[i]["y"]
    X, Y = to5070.transform(lon, lat)                             # into DEM meters
    raster_z = float(dem.sel(x=X, y=Y, method="nearest"))        # nearest pixel value
    print(f"node {i:>5}  ({lat:.5f}, {lon:.5f})  raster {raster_z:6.1f} m")

# ── 5. Independent cross check straight from 3DEP ───────────────────────
# elevation_bycoords asks 3DEP for the exact point height, no raster in the
# middle. Close agreement with step 4 means the sampling is trustworthy.
pts = [(nodes[i]["x"], nodes[i]["y"]) for i in check_ids]
direct = py3dep.elevation_bycoords(pts, crs=4326)
print("\n── Node spot check (3DEP direct) ──")
for i, z in zip(check_ids, np.atleast_1d(direct)):
    print(f"node {i:>5}  3DEP direct {float(z):6.1f} m")

Graph loaded from GitHub raw.
6,749 nodes loaded.
bbox (W,S,E,N): (np.float64(-122.8598), np.float64(38.3423), np.float64(-122.5471), np.float64(38.5313))

── DEM summary ──
CRS      : EPSG:5070
shape    : (3136, 3587) (rows, cols)
pixel    : 8.88 m
elev min : 7.3 m
elev max : 831.9 m
elev mean: 160.1 m

── Node spot check (raster) ──
node     0  (38.41872, -122.71603)  raster   45.1 m
node  3374  (38.43736, -122.76593)  raster   30.6 m

── Node spot check (3DEP direct) ──
node     0  3DEP direct   45.1 m
node  3374  3DEP direct   30.7 m


## 2. Sample elevation onto all 6,749 nodes

Cell 1 proved one tile and two points are trustworthy. Now we read an elevation
for every node in the graph and store it as a new `elev` field.

**Vectorized, not a loop:** instead of sampling nodes one at a time, we hand
xarray the whole batch of coordinates at once and it returns all elevations in a
single pointwise "nearest pixel" lookup. Faster and less code.

**Why reproject again:** same reason as Cell 1. The node coordinates are lon/lat
degrees, the DEM lives in 5070 meters, so we convert the batch before sampling.

**Precision:** we round to 0.1 m. Ten centimeters is far finer than a flood
stage slider will ever care about, and it keeps the numbers short in the graph
file.

**Checks before moving on:** zero NaN values (every node landed on real data),
the graph's elevation spread matches the DEM summary from Cell 1, and nodes 0
and 3374 still read 45.1 and 30.6 m, proving nothing shifted between cells.

In [4]:
# ── Cell 2: sample elevation onto every node ────────────────────────────
# Requires Cell 1 to have run (dem, nodes, lons, lats, to5070 already exist).

import xarray as xr

# ── 1. Reproject ALL node coordinates into the DEM's 5070 meters at once ─
# Transformer.transform happily takes arrays, so this is one vectorized call.
X, Y = to5070.transform(lons, lats)              # arrays of meters, len 6,749

# ── 2. Pointwise "nearest pixel" sample for the whole batch ─────────────
# Wrapping X and Y as DataArrays that share a dim ("pt") tells xarray to pair
# them up: sample #0 uses (X[0], Y[0]), sample #1 uses (X[1], Y[1]), and so on.
# Without the shared dim it would build a full grid, which we do not want.
xs = xr.DataArray(X, dims="pt")
ys = xr.DataArray(Y, dims="pt")
elev = dem.sel(x=xs, y=ys, method="nearest").values.astype(float)  # len 6,749

# ── 3. Sanity: did anything miss the DEM? ───────────────────────────────
n_nan = int(np.isnan(elev).sum())
print(f"sampled {len(elev):,} nodes, {n_nan} NaN")
if n_nan:
    print("WARNING: some nodes fell outside the DEM. Widen PAD in Cell 1.")

# ── 4. Write elev (rounded to 0.1 m) back into the in-memory graph ──────
# We mutate the same nodes list the graph holds, so graph['nodes'] now carries
# elevation. Nothing is exported yet. Cell 4 writes the final file.
for i, z in enumerate(elev):
    nodes[i]["elev"] = round(float(z), 1)

# ── 5. Report the spread and confirm continuity with Cell 1 ─────────────
vals = np.array([n["elev"] for n in nodes])
print("\n── Node elevation spread ──")
print("min :", round(float(vals.min()), 1), "m")
print("max :", round(float(vals.max()), 1), "m")
print("mean:", round(float(vals.mean()), 1), "m")
print("\nnode    0 elev:", nodes[0]["elev"], "m  (Cell 1 said 45.1)")
print("node 3374 elev:", nodes[3374]["elev"], "m  (Cell 1 said 30.6)")

sampled 6,749 nodes, 0 NaN

── Node elevation spread ──
min : 20.6 m
max : 391.0 m
mean: 64.2 m

node    0 elev: 45.1 m  (Cell 1 said 45.1)
node 3374 elev: 30.6 m  (Cell 1 said 30.6)


## 3. Minimum elevation along every edge (the underpass catch)

Node elevations alone miss the scenario that matters most for flooding: a road
that dips **below both of its endpoints**, like an underpass or a creek dip. If
we only knew the two corner heights, that low point would look dry. So for each
of the 16,254 edges we sample elevation along its length and keep the lowest
value as `emin`.

**Densify to ~30 m:** we drop a sample point roughly every 30 m along the edge,
the same spacing the fire model uses. A sag or underpass is almost always longer
than 30 m, so it cannot slip between samples.

**Work in meters:** we reproject each edge into EPSG:5070 (the DEM's own meters)
so the 30 m spacing is even and honest, then sample the DEM directly since it is
already in that projection.

**Straight vs curved edges:** a straight edge is just its two nodes. A curved
edge also carries `geom`, the bend vertices, which we thread between the two
nodes so the sampled line follows the real road shape.

**The payoff check:** we count how many edges have `emin` more than a meter below
their lower endpoint. Those are the sags and underpasses. If that count is a few
hundred, the feature is earning its place. If a geom flip crept in, points would
land far away and show up as NaN, so the NaN count guards that too.

In [5]:
# ── Cell 3: minimum elevation along each edge ───────────────────────────
# Requires Cells 1 and 2 (dem, nodes, edges, to5070 already exist).

import math

edges = graph["edges"]

# ── 1. Densify helper: walk an edge in meters, drop a point every ~30 m ──
def densify_m(coords_m, step=30.0):
    # coords_m: list of (x, y) in meters, ordered along the edge (u -> v)
    out = []
    for (x0, y0), (x1, y1) in zip(coords_m[:-1], coords_m[1:]):
        seg = math.hypot(x1 - x0, y1 - y0)      # segment length in meters
        n = max(1, math.ceil(seg / step))       # how many steps fit
        for k in range(n):                       # include start, skip end (no dupes)
            t = k / n
            out.append((x0 + t * (x1 - x0), y0 + t * (y1 - y0)))
    out.append(coords_m[-1])                     # add the final endpoint once
    return out

# ── 2. Build every sample point, tagged with which edge it belongs to ───
# We collect ALL points into flat lists so we can sample the DEM in one shot,
# then use `owner` to fold the results back to a per-edge minimum.
all_x, all_y, owner = [], [], []
for ei, e in enumerate(edges):
    pu = (nodes[e["u"]]["x"], nodes[e["u"]]["y"])   # start node lon/lat
    pv = (nodes[e["v"]]["x"], nodes[e["v"]]["y"])   # end node lon/lat
    g = e.get("geom")                                # bend vertices, or None
    verts = [pu] + [(p[0], p[1]) for p in g] + [pv] if g else [pu, pv]

    # reproject this edge's vertices into 5070 meters, then densify
    vx, vy = to5070.transform([c[0] for c in verts], [c[1] for c in verts])
    dens = densify_m(list(zip(vx, vy)))

    for x, y in dens:
        all_x.append(x); all_y.append(y); owner.append(ei)

print(f"{len(edges):,} edges -> {len(all_x):,} sample points")

# ── 3. Sample the DEM at every point in one vectorized call ─────────────
PX = xr.DataArray(np.array(all_x), dims="pt")
PY = xr.DataArray(np.array(all_y), dims="pt")
Z  = dem.sel(x=PX, y=PY, method="nearest").values.astype(float)

n_nan = int(np.isnan(Z).sum())
print(f"NaN sample points: {n_nan}  (should be 0; nonzero hints a geom flip)")

# ── 4. Fold points down to one minimum per edge ─────────────────────────
# np.minimum.at walks every point and keeps the smallest value for its edge.
# NaN points are pushed to +inf first so a stray miss cannot poison an edge.
owner = np.array(owner)
Zc = np.where(np.isnan(Z), np.inf, Z)
emin = np.full(len(edges), np.inf)
np.minimum.at(emin, owner, Zc)

# ── 5. Compare each edge's low point to its lower endpoint ──────────────
# lo_end = the shallower of the edge's two node elevations. Where emin sits
# well below that, the road dips between its corners: a sag or underpass.
lo_end = np.array([min(nodes[e["u"]]["elev"], nodes[e["v"]]["elev"]) for e in edges])
dips = np.isfinite(emin) & (emin < lo_end - 1.0)
print(f"edges dipping >1 m below their lower endpoint: {int(dips.sum()):,}")

# ── 6. Write emin (0.1 m) back into each edge; fall back if no sample ────
for ei, e in enumerate(edges):
    z = emin[ei] if np.isfinite(emin[ei]) else lo_end[ei]
    e["emin"] = round(float(z), 1)

vals = np.array([e["emin"] for e in edges])
print("\n── Edge emin spread ──")
print("min :", round(float(vals.min()), 1), "m")
print("max :", round(float(vals.max()), 1), "m")
print("mean:", round(float(vals.mean()), 1), "m")

16,254 edges -> 111,845 sample points
NaN sample points: 0  (should be 0; nonzero hints a geom flip)
edges dipping >1 m below their lower endpoint: 448

── Edge emin spread ──
min : 17.0 m
max : 273.7 m
mean: 61.6 m


## 4. Write elev + emin into the graph and re-export

The in-memory graph now carries everything: `elev` on every node, `emin` on
every edge. Nothing else was touched. This cell serializes it, measures it,
proves it re-parses, and downloads it for a GitHub web-UI upload.

**Compact JSON:** we strip every optional space so the file stays lean. The
routing code does not care about whitespace.

**Why the gzipped size matters:** Cloudflare serves this file gzipped, and the
app lazy-loads it on the first map click. So the gzipped number, not the raw
one, is what a visitor actually waits on. It was ~345 KB before elevation; the
two new fields will grow it, and we want to see by how much.

**No terminal, no git push:** the file downloads to your machine. From there you
replace `graph_santa_rosa.json` in the repo through the GitHub web UI and commit
to `main`, same browser-only flow as always.

In [6]:
# ── Cell 4: re-export the graph with elev + emin ────────────────────────
# Requires Cells 1-3. graph['nodes'] now carry 'elev', graph['edges'] carry
# 'emin'. Everything else in the graph is untouched.

import json, gzip

OUT = "graph_santa_rosa.json"

# ── 1. Serialize compact (no spaces) so the file stays small ────────────
blob = json.dumps(graph, separators=(",", ":")).encode("utf-8")
with open(OUT, "wb") as f:
    f.write(blob)

# ── 2. Measure raw and gzipped size ─────────────────────────────────────
# Cloudflare serves this gzipped and it lazy-loads on first map click, so the
# gzipped number is the one that decides load time.
raw_kb = len(blob) / 1024
gz_kb  = len(gzip.compress(blob, 9)) / 1024
print(f"raw      : {raw_kb:8.1f} KB")
print(f"gzipped  : {gz_kb:8.1f} KB   (was ~345 KB before elevation)")

# ── 3. Verify the file re-parses and carries the new fields ─────────────
check = json.loads(open(OUT).read())
n0 = check["nodes"][0]
e0 = check["edges"][0]
print("\nnode 0 has elev:", "elev" in n0, "->", n0.get("elev"), "m")
print("edge 0 has emin:", "emin" in e0, "->", e0.get("emin"), "m")
print("nodes:", len(check["nodes"]), " edges:", len(check["edges"]))

# ── 4. Download for a GitHub web-UI upload ──────────────────────────────
from google.colab import files
files.download(OUT)
print("\nDownloaded. Replace graph_santa_rosa.json in the repo, commit to main.")

raw      :   1936.1 KB
gzipped  :    389.3 KB   (was ~345 KB before elevation)

node 0 has elev: True -> 45.1 m
edge 0 has emin: True -> 43.2 m
nodes: 6749  edges: 16254


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Downloaded. Replace graph_santa_rosa.json in the repo, commit to main.
